# SUPER SENTAI ALL-IN-ONE PIPELINE (NGŨ HÀNH CHIẾN ĐỘI)
Notebook duy nhất chạy tuần tự từ **A → Z** trên **Google Colab (T4 GPU)** hoặc **IDE Colab/Jupyter Extension**:
1. Cài đặt thư viện & kiểm tra GPU.
2. Khởi tạo vũ trụ (`init_sentai_universe.py`).
3. Tải ảnh nhân vật AI + tách nền `rembg` (`fetch_ai_characters.py`).
4. Tạo **Stock Clips hành động** (`LTX-Video`) lưu thẳng vào `assets/stock_clips/`.
5. Tạo **Diễn xuất cử động khuôn mặt** (`LivePortrait`) lưu thẳng vào `assets/characters/<char_id>/liveportrait_civilian.mp4`.
6. Tạo **Giọng lồng tiếng cảm xúc** (`F5-TTS`, tùy chọn nếu có `ref_voice.wav`).
7. Render phim hoàn chỉnh bằng `sentai_series_engine.py`.

In [ ]:
# BƯỚC 0: CẤU HÌNH CHUNG (CHỈ SỬA Ở ĐÂY)
import os
from pathlib import Path

PROJECT_DIR = Path(".")
EP_FILE = "episodes/ep_001.json"
MODE = "youtube"  # "youtube" (16:9) hoặc "tiktok" (9:16)

# Bật/tắt từng bước theo nhu cầu
RUN_INIT_UNIVERSE = True
RUN_FETCH_AI_CHARS = True
RUN_STOCK_CLIPS = True
RUN_LIVEPORTRAIT = True
RUN_F5TTS_VOICE = False   # Bật True nếu bạn đã đặt file ref_voice.wav vào thư mục dự án
REF_VOICE_WAV = "ref_voice.wav"
RUN_FINAL_RENDER = True

# Kiểm tra môi trường Colab
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    pass

print(f"IN_COLAB={IN_COLAB} | EP_FILE={EP_FILE} | MODE={MODE}")

In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & KIỂM TRA GPU
!pip install -q edge-tts gTTS Pillow numpy pysubs2 mutagen requests imageio-ffmpeg rembg onnxruntime pydub
!pip install -q diffusers transformers accelerate sentencepiece f5-tts

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ĐỒNG BỘ CODE TỪ GITHUB KHI CHẠY TRÊN COLAB
REPO_URL = "https://github.com/nguyenkhanhan123/Test.git"

if IN_COLAB and not Path("sentai_series_engine.py").exists():
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not Path(repo_name).exists():
        !git clone {REPO_URL}
    %cd {repo_name}
    !git pull
elif IN_COLAB and Path(".git").exists():
    !git pull

In [ ]:
# BƯỚC 2: KHỞI TẠO VŨ TRỤ & TÀI NGUYÊN CỐ ĐỊNH
if RUN_INIT_UNIVERSE:
    !python init_sentai_universe.py

In [ ]:
# BƯỚC 3: TẢI ẢNH NHÂN VẬT AI + TÁCH NỀN REMBG
if RUN_FETCH_AI_CHARS:
    !python fetch_ai_characters.py

In [ ]:
# BƯỚC 4: TẠO STOCK CLIPS HÀNH ĐỘNG BẰNG LTX-VIDEO (LƯU THẲNG VÀO assets/stock_clips/)
if RUN_STOCK_CLIPS:
    import gc, torch
    from diffusers import LTXPipeline
    from diffusers.utils import export_to_video

    if not torch.cuda.is_available():
        print("⚠️ Không có GPU CUDA — bỏ qua bước tạo video LTX-Video.")
    else:
        stock_dir = Path("assets/stock_clips")
        stock_dir.mkdir(parents=True, exist_ok=True)
        pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()

        CLIPS = {
            "henshin_fire.mp4": "anime tokusatsu red ranger warrior transforming, crimson metallic armor pieces flying and locking onto body, blazing fire aura sparks, dynamic camera orbit, cinematic action",
            "henshin_water.mp4": "anime tokusatsu blue ranger warrior transforming, cobalt metallic armor assembling with swirling ocean water vortex, glowing visor activation, cinematic action",
            "henshin_wind.mp4": "anime tokusatsu green ranger warrior transforming, emerald aerodynamic armor locking in with green wind cyclone, dramatic tokusatsu lighting",
            "rollcall_explosion.mp4": "five colorful anime tokusatsu super sentai rangers posing heroically in formation, massive colorful smoke explosions erupting behind them, epic anime action",
            "gokai_slash_finisher.mp4": "anime red sentai warrior charging glowing elemental energy sword and unleashing a massive fiery crescent slash wave across battlefield, sparks flying",
            "mecha_combine.mp4": "five giant mechanical elemental beast robots combining in mid-air into one colossal heroic armored mecha titan, glowing golden core, dramatic sky",
            "mecha_sword_finish.mp4": "giant heroic armored mecha robot swinging a colossal glowing energy greatsword, huge explosion in background sky, epic tokusatsu anime",
        }

        for fname, prompt in CLIPS.items():
            out_path = stock_dir / fname
            if out_path.exists():
                print("Skip (đã có):", fname)
                continue
            print("Generating Stock Clip:", fname)
            frames = pipe(
                prompt=prompt,
                negative_prompt="low quality, blurry, distorted, static, watermark, text",
                width=768,
                height=512,
                num_frames=97,
                num_inference_steps=30,
                guidance_scale=3.0,
            ).frames[0]
            export_to_video(frames, str(out_path), fps=24)
            print("Saved:", out_path)

        del pipe
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# BƯỚC 5: TẠO DIỄN XUẤT KHUÔN MẶT BẰNG LIVEPORTRAIT (LƯU VÀO assets/characters/<char_id>/liveportrait_civilian.mp4)
if RUN_LIVEPORTRAIT:
    import shutil, subprocess, torch

    if not torch.cuda.is_available():
        print("⚠️ Không có GPU CUDA — bỏ qua bước LivePortrait.")
    else:
        lp_repo = Path("_tmp_LivePortrait")
        if not lp_repo.exists():
            !git clone -q https://github.com/KwaiVGI/LivePortrait _tmp_LivePortrait
            !pip install -q -r _tmp_LivePortrait/requirements.txt
            !huggingface-cli download KwaiVGI/LivePortrait --local-dir _tmp_LivePortrait/pretrained_weights --exclude "*.git*" "README.md" "docs"

        driving = lp_repo / "assets/examples/driving/d0.mp4"
        char_root = Path("assets/characters")
        for c_dir in sorted(char_root.iterdir()):
            src_img = c_dir / "civilian.png"
            target_mp4 = c_dir / "liveportrait_civilian.mp4"
            if not src_img.exists() or target_mp4.exists():
                continue
            tmp_out = lp_repo / "out_tmp" / c_dir.name
            tmp_out.mkdir(parents=True, exist_ok=True)
            print("Animating LivePortrait:", c_dir.name)
            subprocess.run([
                "python", str(lp_repo / "inference.py"),
                "-s", str(src_img.resolve()),
                "-d", str(driving.resolve()),
                "-o", str(tmp_out.resolve()),
            ], check=True)
            for mp4 in tmp_out.glob("*.mp4"):
                if "concat" not in mp4.name:
                    shutil.copy2(mp4, target_mp4)
                    print("Saved:", target_mp4)
                    break

In [ ]:
# BƯỚC 6: TẠO GIỌNG LỒNG TIẾNG CẢM XÚC F5-TTS (TÙY CHỌN NẾU CÓ ref_voice.wav)
if RUN_F5TTS_VOICE:
    import json, subprocess
    from pydub import AudioSegment

    if not Path(REF_VOICE_WAV).exists():
        print(f"⚠️ Không tìm thấy {REF_VOICE_WAV} — bỏ qua F5-TTS, Engine sẽ tự dùng edge-tts/gTTS.")
    else:
        ep = json.load(open(EP_FILE, encoding="utf-8"))
        ep_code = f"ep_{ep['episode_number']:03d}"
        out_dir = Path("build_output") / ep_code / "audio"
        out_dir.mkdir(parents=True, exist_ok=True)
        ref_text = "Đây là giọng mẫu chuẩn dùng để lồng tiếng cho chiến binh Ngũ Hành Chiến Đội."

        for act in ep["acts"]:
            aid = act["act_id"]
            for idx, sc in enumerate(act["scenes"]):
                sid = f"{aid}_s{idx:03d}"
                wav_out = out_dir / f"{sid}.wav"
                mp3_out = out_dir / f"{sid}.mp3"
                print("F5-TTS:", sid)
                subprocess.run([
                    "f5-tts_infer-cli",
                    "--model", "F5TTS_v1_Base",
                    "--ref_audio", REF_VOICE_WAV,
                    "--ref_text", ref_text,
                    "--gen_text", sc["text"],
                    "--output_dir", str(out_dir),
                    "--output_file", f"{sid}.wav",
                ], check=True)
                AudioSegment.from_wav(wav_out).export(mp3_out, format="mp3", bitrate="192k")
                wav_out.unlink(missing_ok=True)

In [ ]:
# BƯỚC 7: RENDER PHIM HOÀN CHỈNH (YOUTUBE 16:9 HOẶC TIKTOK 9:16)
if RUN_FINAL_RENDER:
    !python -u sentai_series_engine.py --ep {EP_FILE} --mode {MODE} --force

In [ ]:
# BƯỚC 8: XUẤT / TẢI FILE PHIM THÀNH PHẨM
import json
ep = json.load(open(EP_FILE, encoding="utf-8"))
ep_code = f"ep_{ep['episode_number']:03d}"
final_mp4 = Path(f"build_output/NguHanhChienDoi_{ep_code}_{MODE.upper()}_FULL.mp4")
print("File phim:", final_mp4.resolve(), "| Tồn tại:", final_mp4.exists())

if IN_COLAB and final_mp4.exists():
    from google.colab import files
    files.download(str(final_mp4))